# 07 — Train Transformer Pricing Network

Train three separate Transformer instances with identical architecture:
- **Experiment A:** 50,000 Heston paths only
- **Experiment B:** 50,000 GAN paths only
- **Experiment C:** 25,000 Heston + 25,000 GAN mixed

All use: Adam lr=1e-4, MSE loss, early stopping (patience 10), ReduceLROnPlateau.

In [ ]:
import sys, os
import numpy as np
import torch
sys.path.insert(0, os.path.abspath('..'))

from src.models.transformer import build_transformer
from src.training.train_transformer import train_transformer
from src.data.dataset import SimulatedPricingDataset

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
CHECKPOINT_BASE = os.path.join('..', 'checkpoints', 'transformer')

device = torch.device('cuda' if torch.cuda.is_available() else
                       'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load data
train_seqs = np.load(os.path.join(PROCESSED_DIR, 'train_sequences.npy'))

h_contracts = np.load(os.path.join(PROCESSED_DIR, 'heston_contracts.npy'))
h_labels = np.load(os.path.join(PROCESSED_DIR, 'heston_labels.npy'))

g_contracts_path = os.path.join(PROCESSED_DIR, 'gan_contracts.npy')
has_gan = os.path.exists(g_contracts_path)
if has_gan:
    g_contracts = np.load(g_contracts_path)
    g_labels = np.load(os.path.join(PROCESSED_DIR, 'gan_labels.npy'))
    print(f'GAN data loaded: {len(g_labels)} samples')

print(f'Training sequences: {train_seqs.shape}')
print(f'Heston data: {len(h_labels)} samples')

## Experiment A: Heston Only

In [ ]:
ds_a = SimulatedPricingDataset(train_seqs, h_contracts, h_labels)
model_a = build_transformer()

model_a, history_a = train_transformer(
    model_a, ds_a,
    batch_size=256, max_epochs=100, lr=1e-4,
    patience=10, lr_patience=5, lr_factor=0.5,
    device=device,
    checkpoint_dir=os.path.join(CHECKPOINT_BASE, 'experiment_a'),
    experiment_name='Experiment A (Heston)',
)

## Experiment B: GAN Only

In [ ]:
if has_gan:
    ds_b = SimulatedPricingDataset(train_seqs, g_contracts, g_labels)
    model_b = build_transformer()
    
    model_b, history_b = train_transformer(
        model_b, ds_b,
        batch_size=256, max_epochs=100, lr=1e-4,
        patience=10, lr_patience=5, lr_factor=0.5,
        device=device,
        checkpoint_dir=os.path.join(CHECKPOINT_BASE, 'experiment_b'),
        experiment_name='Experiment B (GAN)',
    )
else:
    print('Skipping Experiment B (no GAN data).')

## Experiment C: Mixed (25K Heston + 25K GAN)

In [ ]:
if has_gan:
    mixed_contracts = np.concatenate([h_contracts[:25000], g_contracts[:25000]], axis=0)
    mixed_labels = np.concatenate([h_labels[:25000], g_labels[:25000]], axis=0)
    ds_c = SimulatedPricingDataset(train_seqs, mixed_contracts, mixed_labels)
    
    model_c = build_transformer()
    
    model_c, history_c = train_transformer(
        model_c, ds_c,
        batch_size=256, max_epochs=100, lr=1e-4,
        patience=10, lr_patience=5, lr_factor=0.5,
        device=device,
        checkpoint_dir=os.path.join(CHECKPOINT_BASE, 'experiment_c'),
        experiment_name='Experiment C (Mixed)',
    )
else:
    print('Skipping Experiment C (no GAN data).')

## Training Curves Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, hist, color in [('A: Heston', history_a, 'blue')]:
    axes[0].plot(hist['train_loss'], color=color, alpha=0.7, label=f'{name} (train)')
    axes[0].plot(hist['val_loss'], color=color, linestyle='--', label=f'{name} (val)')

if has_gan:
    for name, hist, color in [('B: GAN', history_b, 'green'), ('C: Mixed', history_c, 'red')]:
        axes[0].plot(hist['train_loss'], color=color, alpha=0.7, label=f'{name} (train)')
        axes[0].plot(hist['val_loss'], color=color, linestyle='--', label=f'{name} (val)')

axes[0].set_title('Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend(fontsize=8)
axes[0].set_yscale('log')

# LR schedule
axes[1].plot(history_a['lr'], label='A: Heston')
if has_gan:
    axes[1].plot(history_b['lr'], label='B: GAN')
    axes[1].plot(history_c['lr'], label='C: Mixed')
axes[1].set_title('Learning Rate Schedule')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('LR')
axes[1].set_yscale('log')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'figures', 'transformer_training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()